In [1]:
# 1. 필요한 라이브러리를 불러온다.
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    pipeline
)
from peft import PeftModel

d:\dev\workspace\ai\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. 저장된 LoRA를 불러온다.
base_model_id = "beomi/kcbert-base" 
lora_model_path = "./saved_models/lora_sentiment/final_model"

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=2
)

tokenizer = AutoTokenizer.from_pretrained(lora_model_path)
lora_model = PeftModel.from_pretrained(base_model, lora_model_path)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1826.06it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [3]:
# 3. transformers 파이프라인을 구성한다.
pipe = pipeline(
    task="text-classification",
    model=lora_model,
    tokenizer=tokenizer,
)

In [5]:
# 4. 예측에 활용될 데이터를 준비한다.
texts = [
    # 긍정 데이터
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.",
    "스토리는 평범했지만 연출 덕분에 재미있었어요.",
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.",
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.",
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.",
    
    # 부정 데이터
    "이야기가 늘어져서 중간부터 집중이 안 됐어요.",
    "연출이 과해서 오히려 몰입을 방해했어요.",
    "캐릭터 행동이 이해되지 않아서 답답했어요.",
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.",
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요."
]

In [6]:
# 5. 준비된 데이터를 활용해 예측한다.
predicts = pipe(texts)
print(predicts)

[{'label': 'LABEL_0', 'score': 0.5844194293022156}, {'label': 'LABEL_0', 'score': 0.5225616097450256}, {'label': 'LABEL_1', 'score': 0.5586998462677002}, {'label': 'LABEL_1', 'score': 0.5749966502189636}, {'label': 'LABEL_1', 'score': 0.5222266912460327}, {'label': 'LABEL_0', 'score': 0.5180022120475769}, {'label': 'LABEL_0', 'score': 0.584254801273346}, {'label': 'LABEL_1', 'score': 0.5720983147621155}, {'label': 'LABEL_1', 'score': 0.5496841669082642}, {'label': 'LABEL_1', 'score': 0.5400035381317139}]


In [7]:
for text, pred in zip(texts, predicts):
    if pred['label'] == 'LABEL_1':
        result = "긍정"
    else:
        result = "부정"
    print(f'입력 데이터: "{text}", 감성 분석 결과: {result}')

입력 데이터: "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.", 감성 분석 결과: 부정
입력 데이터: "스토리는 평범했지만 연출 덕분에 재미있었어요.", 감성 분석 결과: 부정
입력 데이터: "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.", 감성 분석 결과: 긍정
입력 데이터: "큰 기대 없이 봤는데 생각보다 괜찮았어요.", 감성 분석 결과: 긍정
입력 데이터: "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.", 감성 분석 결과: 긍정
입력 데이터: "이야기가 늘어져서 중간부터 집중이 안 됐어요.", 감성 분석 결과: 부정
입력 데이터: "연출이 과해서 오히려 몰입을 방해했어요.", 감성 분석 결과: 부정
입력 데이터: "캐릭터 행동이 이해되지 않아서 답답했어요.", 감성 분석 결과: 긍정
입력 데이터: "분위기는 잡으려는 것 같은데 내용이 부족했어요.", 감성 분석 결과: 긍정
입력 데이터: "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요.", 감성 분석 결과: 긍정
